Lấy tất cả bài viết trong ngày

In [31]:
import requests
from bs4 import BeautifulSoup
from datetime import date
import time
import random
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Cấu hình

In [ ]:
today = date.today().strftime("%Y-%m-%d")
headers = {"User-Agent": "Mozilla/5.0"}

all_links = []
page = 1

# 2. Lấy danh sách link

In [42]:
print(f"=== LINK NGÀY {today} ===")
while True:
    if page == 1:
        url = f"https://dantri.com.vn/thoi-su/from/{today}/to/{today}.htm"
    else:
        url = f"https://dantri.com.vn/thoi-su/from/{today}/to/{today}/trang-{page}.htm"

    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = soup.find_all(
        "article",
        class_="article-item",
        attrs={"data-content-name": f"category-timeline_page_{page}"}
    )

    if not articles:
        break

    for art in articles:
        link = art.get("data-content-target")
        if link:
            if link.startswith("/"):
                link = "https://dantri.com.vn" + link
                all_links.append(link)
    page += 1
print("\nTỔNG LINK:", len(all_links))
for l in all_links:
    print(l)


=== LINK NGÀY 2026-01-28 ===

TỔNG LINK: 11
https://dantri.com.vn/thoi-su/du-an-tram-ty-nhu-dong-sat-vun-80000-tan-quang-mac-ket-20260128085336568.htm
https://dantri.com.vn/thoi-su/nguoi-phu-nu-lang-son-nho-cong-an-tim-22-ty-dong-chuyen-nham-20260128085820680.htm
https://dantri.com.vn/thoi-su/phu-nhan-hai-nuoc-viet-lao-ve-ninh-binh-tham-den-tho-cong-chua-nhoi-hoa-20260128084238841.htm
https://dantri.com.vn/thoi-su/xe-dau-keo-lien-tuc-chuyen-lan-chen-ep-o-to-o-tphcm-20260128081801398.htm
https://dantri.com.vn/thoi-su/dong-nai-khoi-cong-thong-xe-ky-thuat-5-du-an-chao-mung-ngay-thanh-lap-dang-20260127161420505.htm
https://dantri.com.vn/thoi-su/thanh-nien-tu-vong-tren-cau-o-tphcm-20260128032720476.htm
https://dantri.com.vn/thoi-su/tphcm-tinh-cach-chong-ngap-moi-20260127221951231.htm
https://dantri.com.vn/thoi-su/viet-nam-keu-goi-cac-nuoc-som-ky-va-phe-chuan-cong-uoc-ha-noi-20260128064001579.htm
https://dantri.com.vn/thoi-su/ba-dot-pha-chien-luoc-tao-su-chuyen-bien-toan-dien-cho-dat-nuoc-20

# 3. Hàm crawl

In [33]:
def crawl_article(url):
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    # ===== TITLE =====
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # ===== BODY =====
    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text:
            paragraphs.append(text)

    body = "\n".join(paragraphs)

    return {
        "url": url,
        "title": title,
        "content": body
    }

In [34]:
print("\n=== TEST 1 BÀI ===")
test_url = all_links[2]
data = crawl_article(test_url)

print("URL:", data["url"])
print("TITLE:", data["title"])
print("CONTENT:")
print(data["content"])


=== TEST 1 BÀI ===
URL: https://dantri.com.vn/thoi-su/phu-nhan-hai-nuoc-viet-lao-ve-ninh-binh-tham-den-tho-cong-chua-nhoi-hoa-20260128084238841.htm
TITLE: Phu nhân hai nước Việt - Lào về Ninh Bình thăm Đền thờ Công chúa Nhồi Hoa
CONTENT:
Phu nhân Ngô Phương Ly và Phu nhân Naly Sisoulith tham gia hoạt động giao lưu với các nghệ nhân tiêu biểu một số làng nghề của tỉnh Ninh Bình (Ảnh: Đại Nghĩa - TTXVN).
Ngày 27/1, trong khuôn khổ chuyến thăm cấp Nhà nước tới Việt Nam của Đoàn đại biểu cấp cao Đảng, Nhà nước Lào, bà Ngô Phương Ly, Phu nhân Tổng Bí thư Đảng Cộng sản Việt Nam, cùng bà Naly Sisoulith, Phu nhân Tổng Bí thư Ban Chấp hành Trung ương Đảng Nhân dân Cách mạng Lào, Chủ tịch nước Cộng hòa Dân chủ Nhân dân Lào, đã đến thăm và có các hoạt động tại tỉnh Ninh Bình.
Đoàn đã tới thăm, dâng hương tại Đền thờ Công chúa Nhồi Hoa (thôn Thái Sơn, phường Tây Hoa Lư) là di tích lịch sử, văn hóa gắn với mối quan hệ hữu nghị Việt Nam - Lào.
Theo sử liệu, dưới triều vua Lê Thánh Tông, trong cuộc 

# 4. Cấu hình Splitter

In [37]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",   # đoạn
        "\n",     # dòng
        ".", "!", "?", ":", ";", ","
    ]
)
dulieu = splitter.split_text(data["content"])
print("Số chunk:", len(dulieu))
for i, chunk in enumerate(dulieu):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)


Số chunk: 8

--- CHUNK 0 ---
Phu nhân Ngô Phương Ly và Phu nhân Naly Sisoulith tham gia hoạt động giao lưu với các nghệ nhân tiêu biểu một số làng nghề của tỉnh Ninh Bình (Ảnh: Đại Nghĩa - TTXVN).
Ngày 27/1, trong khuôn khổ chuyến thăm cấp Nhà nước tới Việt Nam của Đoàn đại biểu cấp cao Đảng, Nhà nước Lào, bà Ngô Phương Ly, Phu nhân Tổng Bí thư Đảng Cộng sản Việt Nam, cùng bà Naly Sisoulith, Phu nhân Tổng Bí thư Ban Chấp hành Trung ương Đảng Nhân dân Cách mạng Lào, Chủ tịch nước Cộng hòa Dân chủ Nhân dân Lào, đã đến thăm và có các hoạt động tại tỉnh Ninh Bình.
Đoàn đã tới thăm, dâng hương tại Đền thờ Công chúa Nhồi Hoa (thôn Thái Sơn, phường Tây Hoa Lư) là di tích lịch sử, văn hóa gắn với mối quan hệ hữu nghị Việt Nam - Lào.

--- CHUNK 1 ---
Theo sử liệu, dưới triều vua Lê Thánh Tông, trong cuộc kháng chiến chống quân Chiêm Thành xâm lược, triều đình và nhân dân Đại Việt nhận được sự hỗ trợ của Vương quốc Lào. Nhà vua Lào đã cử Công chúa Nhồi Hoa cùng một số tướng sĩ và đội voi chiến s

# 5. Xử lý hàng loạt

In [36]:
final_data = []
print("\n=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===")

for i, link in enumerate(all_links):
    print(f"[{i+1}/{len(all_links)}] Đang xử lý: {link}")

    article_data = crawl_article(link)

    if article_data and article_data["content"]:
        # Split text
        chunks = splitter.split_text(article_data["content"])

        # Lưu kết quả
        final_data.append({
            "url": link,
            "title": article_data["title"],
            "chunks": chunks
        })
        print(f"  -> OK: {len(chunks)} chunks")
    else:
        print("  -> Bỏ qua (không lấy được nội dung)")

    # Rate limiting
    sleep_time = random.uniform(1, 2)
    time.sleep(sleep_time)

print(f"\n=== HOÀN THÀNH ===")
print(f"Đã xử lý thành công: {len(final_data)} bài.")



=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===
[1/11] Đang xử lý: https://dantri.com.vn/thoi-su/du-an-tram-ty-nhu-dong-sat-vun-80000-tan-quang-mac-ket-20260128085336568.htm
  -> OK: 4 chunks
[2/11] Đang xử lý: https://dantri.com.vn/thoi-su/nguoi-phu-nu-lang-son-nho-cong-an-tim-22-ty-dong-chuyen-nham-20260128085820680.htm
  -> OK: 2 chunks
[3/11] Đang xử lý: https://dantri.com.vn/thoi-su/phu-nhan-hai-nuoc-viet-lao-ve-ninh-binh-tham-den-tho-cong-chua-nhoi-hoa-20260128084238841.htm
  -> OK: 8 chunks
[4/11] Đang xử lý: https://dantri.com.vn/thoi-su/xe-dau-keo-lien-tuc-chuyen-lan-chen-ep-o-to-o-tphcm-20260128081801398.htm
  -> OK: 2 chunks
[5/11] Đang xử lý: https://dantri.com.vn/thoi-su/dong-nai-khoi-cong-thong-xe-ky-thuat-5-du-an-chao-mung-ngay-thanh-lap-dang-20260127161420505.htm
  -> OK: 3 chunks
[6/11] Đang xử lý: https://dantri.com.vn/thoi-su/thanh-nien-tu-vong-tren-cau-o-tphcm-20260128032720476.htm
  -> OK: 2 chunks
[7/11] Đang xử lý: https://dantri.com.vn/thoi-su/tphcm-tinh-cach-chong-nga

In [ ]:
# 6. Lưu file JSON
output_file = "dulieu.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)
print(f"Đã lưu dữ liệu vào file: {output_file}")